# Chapter 11: Memory in Agents + MCP
## Topic 4: Building a Minimal MCP Server Wrapping Your Tools

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 3 established MCP conceptually — what it standardizes, why it matters, and why this project's current N=1 agent scale doesn't yet justify adopting it in production. This topic exists specifically as the hands-on complement to that conceptual understanding: build a genuine, working MCP server wrapping this project's own tools, using the real, official MCP Python SDK, to see the protocol's actual mechanics concretely rather than only reasoning about them abstractly.
- The core exercise this topic walks through: taking a tool this notebook series has already designed carefully (`check_sender_history`, Chapter 11 Topic 2 — chosen specifically because Topic 3 identified it as an already well-designed, minimally-coupled candidate) and exposing it through an MCP server, using the real SDK's actual conventions, rather than continuing to reason about MCP only through the simulated registration mechanism Topic 3 built to illustrate the concept.
- Why this distinction matters: Topic 3's `generic_dispatch` function was a deliberately simplified illustration of the *idea* behind protocol-based tool discovery. This topic uses the actual, real MCP SDK (`mcp`, published by Anthropic) — meaning the server built here would genuinely work with any real, compliant MCP client, not just the illustrative example built for one notebook's own demonstration purposes.


### 2. Internal Working — Step by Step

**Building a minimal MCP server with the real SDK, step by step:**

1. **Create a `FastMCP` server instance** — this is the SDK's high-level, decorator-based interface for building an MCP server without needing to hand-write the full protocol's lower-level message handling directly; it exists specifically to make exactly this kind of tool-wrapping exercise straightforward.
2. **Decorate a real Python function with `@mcp.tool()`** — this single decorator does the work Chapter 10 Topic 4 described as writing a tool schema by hand (name, description, input schema): the SDK automatically derives the tool's schema from the function's own signature, type hints, and docstring, rather than requiring a separately hand-maintained schema dictionary the way this project's current direct-integration approach does.
3. **The underlying tool logic itself is unchanged** — this is the concrete, working proof of Chapter 10 and Chapter 11's own repeated design principle: a well-designed tool function (structured output, clear found/not-found distinction, curated fields) is exactly what makes wrapping it in MCP straightforward, since the function itself needs no rework at all — only a decorator and, where useful, a docstring refined for the schema-generation the SDK performs automatically.
4. **The server, once defined, can be queried by any compatible client to discover its available tools** — this is the concrete mechanism behind Topic 3's abstract claim about protocol-based discovery: a real MCP client can call `list_tools()` and receive the schema the SDK automatically derived, without that client needing any advance, hardcoded knowledge of what tools this specific server happens to offer.


### 3. How This Is Implemented in This Project

- The server built in this topic wraps `check_sender_history` (Chapter 11 Topic 2's real, tested, CSV-backed implementation) — reusing the exact same underlying function and persistent store already built and verified against real project data, rather than reimplementing the tool's logic specifically for this MCP exercise.
- Following Topic 3's own conclusion, this server is built as a genuine learning and readiness exercise — demonstrating the mechanism works correctly and that this project's tools are indeed MCP-ready by design — without this notebook claiming or recommending that this server should actually replace this project's current, correct, direct-integration production approach at its present N=1 agent scale.
- The `@mcp.tool()` decorator's automatic schema derivation from the function's docstring and type hints is a direct, concrete validation of Chapter 10 Topic 4's schema-writing principles: a docstring written with a clear when-to-call description and well-typed parameters (following this notebook series' own established standards) produces a genuinely good, automatically-derived MCP schema with no additional manual schema-writing work required at all.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Automatic schema derivation is powerful, but only as good as the underlying function's own docstring and type hints — garbage in, garbage out, applies directly here.** A tool function with a vague docstring or loosely-typed parameters (missing exactly the when-to-call and when-not-to-call guidance Chapter 10 Topic 4 measured as having a real, quantified effect on triggering accuracy) will produce an equally vague, automatically-derived MCP schema — the SDK removes the mechanical burden of writing the schema by hand, but doesn't remove the responsibility for writing genuinely good, clear tool documentation in the first place.
- **A `FastMCP` server, once actually deployed (as opposed to this notebook's in-process demonstration), becomes a genuinely separate running process or service** — with its own startup, shutdown, health-check, and error-handling concerns distinct from this project's current single-process `run_agent()` execution, exactly the new operational surface Topic 3 warned MCP adoption introduces.
- **Debugging a tool wrapped in MCP requires understanding both the underlying function's own logic (Chapter 11 Topic 2's design) and the protocol layer wrapping it** — a bug could live in either layer, and correctly attributing which one requires checking whether the underlying function behaves correctly when called directly (bypassing MCP entirely) versus whether the protocol-level request/response handling itself is the source of an issue.
- **Security considerations from Topic 3 become concrete here:** even in this notebook's simplified, in-process demonstration, a genuinely deployed MCP server exposing `check_sender_history` would need real access-control and input-validation discipline at the server boundary — the same dispatch-layer validation principle from Chapter 10 Topic 1, now needing to be implemented at the MCP server's own request-handling layer rather than within a single trusted Python process.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **`FastMCP`'s high-level, decorator-based interface vs the SDK's lower-level protocol primitives:** `FastMCP` is the right choice for straightforward tool-wrapping exercises like this one, trading some low-level control for significantly less boilerplate — the lower-level primitives exist for cases needing finer control over protocol behavior than the decorator-based interface provides, not needed for this project's current, simple wrapping needs.
- **Reusing the exact existing tool function vs writing a new, MCP-specific version:** reusing Chapter 11 Topic 2's real, tested function (this topic's actual approach) validates that a well-designed tool genuinely requires no rework to become MCP-compatible — a strong, concrete argument for this notebook series' repeated design-quality principles, versus writing a parallel, MCP-specific implementation that would risk the two versions drifting out of sync with each other over time.
- **Whether to wrap one tool or several in this initial exercise:** starting with a single, well-understood tool (`check_sender_history`) keeps this exercise focused on the mechanism itself; wrapping this project's full multi-tool set (Chapter 10 Topic 6) in MCP would be a natural, straightforward extension of the same pattern demonstrated here, but isn't necessary to prove the underlying concept works.


### 6. Alternatives and When to Use Each

- **`FastMCP`'s decorator-based approach (this topic's actual approach):** the right choice for straightforward tool-wrapping, minimizing boilerplate and directly leveraging a function's existing docstring and type hints for automatic schema generation.
- **The SDK's lower-level server primitives:** worth using specifically when finer control over protocol-level behavior is genuinely needed — not required for this project's current, simple tool-wrapping needs.
- **Continuing with direct integration only, treating this topic purely as a learning exercise (this project's actual current production stance, per Topic 3's conclusion):** the correct real-world choice at this project's current N=1 agent scale, even while this topic's exercise demonstrates the mechanism works and confirms this project's tools are MCP-ready by design.


### 7. Common Mistakes and Production Failures

- Assuming automatic schema derivation removes the need for careful, well-written docstrings and type hints — a vague docstring produces an equally vague, automatically-derived schema, reproducing exactly the triggering-reliability problems Chapter 10 Topic 4 measured concretely.
- Reimplementing tool logic specifically for MCP-wrapping rather than reusing the existing, already-tested implementation, risking the two versions drifting apart over time as one gets updated without the other.
- Treating a successful in-process demonstration (like this notebook's own exercise) as equivalent to a genuinely production-ready, deployed MCP server, without accounting for the real operational concerns (process management, security, monitoring) a genuine deployment would introduce.
- Building and deploying an MCP server preemptively based on this exercise alone, without Topic 3's concrete adoption trigger actually being met in this project's real production context.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What does the `@mcp.tool()` decorator actually do?
  A: It registers a regular Python function as an MCP-accessible tool, automatically deriving the tool's schema (name, description, parameter types) from the function's own signature, type hints, and docstring — removing the need to separately hand-write a schema dictionary the way this project's direct-integration approach currently requires.

- Q: Why is `check_sender_history` a good first candidate for this MCP-wrapping exercise?
  A: It's already well-designed by this notebook series' own established principles — structured output, a clear found/not-found (here, `is_repeat_sender`) distinction, curated fields, minimal coupling to project-specific internal conventions — meaning it requires essentially no rework to expose safely and usefully through a standardized protocol.

**Intermediate**

- Q: Why does the quality of a tool function's docstring and type hints matter even more once MCP's automatic schema derivation is being used?
  A: The derived schema is only as good as what it's derived from — a vague docstring lacking explicit when-to-call and when-not-to-call guidance produces an equally vague, automatically-generated schema, reproducing the same measured triggering-reliability problems Chapter 10 Topic 4 demonstrated concretely for hand-written schemas. Automatic derivation removes the mechanical writing burden, not the responsibility for writing genuinely clear, complete tool documentation.

- Q: This exercise reuses Chapter 11 Topic 2's exact, already-tested `check_sender_history` function rather than writing a new version. Why does this matter?
  A: It directly validates the claim that a well-designed tool requires no rework to become MCP-compatible — proving the design principles established throughout Chapters 10-11 aren't just abstractly good practice, but concretely produce tools ready for protocol-based exposure without modification. Writing a separate, parallel MCP-specific version instead would risk the two implementations drifting out of sync as one is updated without the other.

**Advanced**

- Q: A teammate suggests this notebook's working MCP server demonstration means the project should deploy it to production immediately. How do you respond?
  A: A successful in-process demonstration proves the *mechanism* works and that this project's tools are MCP-ready by design — it does not by itself establish that MCP adoption is currently justified in production, which depends on Topic 3's concrete triggers (a genuine second consumer, a process-isolation need, cross-team reuse) actually being met. A genuinely deployed server also introduces real operational concerns — process management, security boundaries, monitoring — that this notebook's simplified, in-process exercise doesn't need to address, and that would need real, deliberate design work before any production deployment.

- Q: How would you validate that an automatically-derived MCP schema is actually as effective as a hand-written one, using this notebook series' own established methodology?
  A: Apply the exact same evidence-based validation discipline Chapter 10 Topic 4 established for hand-written schemas — build a labeled test set of realistic inputs with known-correct triggering decisions, measure triggering accuracy using the automatically-derived schema, and compare against the hand-written schema's previously-measured accuracy on the same test set. Automatic derivation is a convenience for schema *construction*, not an exemption from the *validation* discipline that applies to any schema, however it was produced.

**Scenario-based**

- Q: You're wrapping a second tool in MCP and notice its existing docstring is much less detailed than `check_sender_history`'s. What would you do before wrapping it?
  A: Improve the docstring first, applying Chapter 10 Topic 4's schema-writing principles directly — explicit mechanism, explicit when-to-call condition, and critically, an explicit when-NOT-to-call boundary — before relying on automatic derivation to produce a good schema from it. Wrapping a poorly-documented tool as-is would produce an equally poor, automatically-derived schema, reproducing exactly the measured triggering-reliability problem this notebook series has repeatedly demonstrated for underspecified tool descriptions.

**Follow-up questions to expect:**

- "What would you add to this minimal server before considering it genuinely production-ready?"
- "How would you test that the automatically-derived schema for a new tool is actually correct, before relying on it?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Automatic schema derivation from code (docstrings, type hints) is a broader pattern in software tooling, not unique to MCP** — API documentation generators, OpenAPI/Swagger schema generation from typed function signatures, and similar tools all apply this same underlying idea: well-structured, well-typed, well-documented code can mechanically produce a usable interface description, reducing duplicated manual effort between the code and its documentation.
- **This exercise is a direct, concrete test of a design principle asserted repeatedly throughout this notebook series** — "well-designed tools are inherently more reusable, portable, and maintainable" — rather than just an assertion, this topic actually demonstrates it by successfully wrapping an already-existing, unmodified tool in a completely different exposure mechanism.
- **This topic sets up Topic 5 directly:** having a genuine, working MCP server is the necessary prerequisite for the next topic's exercise — actually connecting a client (an agent) to it end to end, completing the full loop from Topic 3's conceptual understanding through this topic's server-building exercise to a genuinely working, tested client-server interaction.

### 10. Quick Revision Summary (for last-minute interview prep)

> Building a minimal MCP server means taking an already-well-designed tool function (`check_sender_history`, reused exactly as built and tested in Topic 2) and exposing it through the real MCP Python SDK's `FastMCP` interface, using the `@mcp.tool()` decorator to automatically derive the tool's schema from the function's own signature, type hints, and docstring — rather than hand-writing a separate schema dictionary the way this project's current direct-integration approach requires. This exercise concretely validates this notebook series' own repeated design principle: a tool built with structured output, clear found/not-found distinctions, and curated fields requires no rework at all to become MCP-compatible. Automatic schema derivation removes the mechanical burden of schema-writing, but not the responsibility for writing genuinely clear, complete docstrings — a vague docstring still produces an equally vague, automatically-derived schema, reproducing exactly the triggering-reliability problems Chapter 10 Topic 4 measured for hand-written schemas. This remains a learning and readiness exercise, not a production recommendation — Topic 3's concrete adoption triggers still need to be genuinely met before this project's actual production system should move away from its current, correct, direct-integration approach at N=1 agent scale.


### Module 1: The Real, Tested Tool Function, Unchanged

Reuse Chapter 11 Topic 2's exact check_sender_history implementation -- no modifications, proving it's already MCP-ready by design.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: The real, already-tested tool function -- UNCHANGED
# ------------------------------------------------------------------

import csv
import os
from datetime import datetime

SENDER_HISTORY_PATH = "/home/claude/build/sender_history_mcp_demo.csv"
FIELDNAMES = ["sender_email", "first_seen_date", "total_email_count",
              "last_topic", "has_unresolved_issue", "last_contact_date"]


def _read_all_records() -> dict:
    if not os.path.exists(SENDER_HISTORY_PATH):
        return {}
    records = {}
    with open(SENDER_HISTORY_PATH, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row["total_email_count"] = int(row["total_email_count"])
            row["has_unresolved_issue"] = row["has_unresolved_issue"] == "True"
            records[row["sender_email"]] = row
    return records


def _write_all_records(records: dict):
    with open(SENDER_HISTORY_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        for record in records.values():
            writer.writerow(record)


def check_sender_history_impl(sender_email: str) -> dict:
    """The EXACT same real function from Chapter 11 Topic 2 -- proven,
    tested, unmodified. This is what we will wrap in MCP below."""
    records = _read_all_records()
    record = records.get(sender_email.strip().lower())
    if record is None:
        return {"sender_email": sender_email, "is_repeat_sender": False}
    return {"sender_email": sender_email, "is_repeat_sender": True, **record}


def write_sender_interaction_impl(sender_email: str, topic: str, unresolved: bool):
    key = sender_email.strip().lower()
    records = _read_all_records()
    today = datetime.now().strftime("%Y-%m-%d")
    if key not in records:
        records[key] = {"sender_email": key, "first_seen_date": today, "total_email_count": 1,
                          "last_topic": topic, "has_unresolved_issue": unresolved, "last_contact_date": today}
    else:
        existing = records[key]
        existing["total_email_count"] += 1
        existing["last_topic"] = topic
        existing["has_unresolved_issue"] = unresolved
        existing["last_contact_date"] = today
    _write_all_records(records)


# pre-populate a real record for the demo
if os.path.exists(SENDER_HISTORY_PATH):
    os.remove(SENDER_HISTORY_PATH)
write_sender_interaction_impl("shobha.chopra@email.com", "premature withdrawal penalty", unresolved=True)

print("=" * 70)
print("MODULE 1: REAL, ALREADY-TESTED FUNCTION -- ZERO MODIFICATIONS")
print("=" * 70)
direct_call_result = check_sender_history_impl("shobha.chopra@email.com")
print(f"\nCalling the function DIRECTLY (no MCP involved yet): {direct_call_result}")
print("\nThis is the SAME function, unmodified, that Module 2 will wrap")
print("in a real MCP server using the official SDK.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL, ALREADY-TESTED FUNCTION -- ZERO MODIFICATIONS

Calling the function DIRECTLY (no MCP involved yet): {'sender_email': 'shobha.chopra@email.com', 'is_repeat_sender': True, 'first_seen_date': '2026-07-09', 'total_email_count': 1, 'last_topic': 'premature withdrawal penalty', 'has_unresolved_issue': True, 'last_contact_date': '2026-07-09'}

This is the SAME function, unmodified, that Module 2 will wrap
in a real MCP server using the official SDK.

Module 1 complete. Run Module 2 next.


### Module 2: A Real MCP Server, Built With the Official SDK

Use the genuine mcp package (Anthropic's official Python SDK) to wrap check_sender_history_impl with @mcp.tool() -- a real, working MCP server, not a simulation.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: A REAL MCP server, built with the official Anthropic SDK
# ------------------------------------------------------------------

from mcp.server.fastmcp import FastMCP

# a REAL FastMCP server instance
mcp_server = FastMCP("sender-history-server")


@mcp_server.tool()
def check_sender_history(sender_email: str) -> dict:
    """Look up whether this sender has contacted us before, and if so,
    what their last topic was and whether it was resolved.
    Call this at the start of handling a new email to get context
    about the sender's history with us.
    Do NOT call this for a sender_email you have not yet verified
    belongs to the actual email being processed.
    """
    return check_sender_history_impl(sender_email)


print("=" * 70)
print("MODULE 2: REAL MCP SERVER BUILT (official Anthropic SDK)")
print("=" * 70)
print(f"\nServer name: {mcp_server.name}")

# inspect what schema the SDK ACTUALLY, AUTOMATICALLY derived from
# our function's signature and docstring -- this is REAL, not simulated
import asyncio

async def inspect_server_tools():
    tools = await mcp_server.list_tools()
    return tools

tools = asyncio.run(inspect_server_tools())

print(f"\nTools the server reports (via REAL list_tools() call): {len(tools)}")
for tool in tools:
    print(f"\n  Tool name: {tool.name}")
    print(f"  Description (AUTOMATICALLY derived from our docstring):")
    print(f"    {tool.description[:200]}...")
    print(f"  Input schema (AUTOMATICALLY derived from our type hints):")
    print(f"    {tool.inputSchema}")

print("\nThis schema was NEVER hand-written -- it was derived entirely")
print("from check_sender_history's own docstring and type hint (sender_email: str),")
print("exactly Module 1's real, unmodified function -- concrete proof that a")
print("well-designed tool needs NO REWORK to become MCP-compatible.")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: REAL MCP SERVER BUILT (official Anthropic SDK)

Server name: sender-history-server

Tools the server reports (via REAL list_tools() call): 1

  Tool name: check_sender_history
  Description (AUTOMATICALLY derived from our docstring):
    Look up whether this sender has contacted us before, and if so,
    what their last topic was and whether it was resolved.
    Call this at the start of handling a new email to get context
    about t...
  Input schema (AUTOMATICALLY derived from our type hints):
    {'properties': {'sender_email': {'title': 'Sender Email', 'type': 'string'}}, 'required': ['sender_email'], 'title': 'check_sender_historyArguments', 'type': 'object'}

This schema was NEVER hand-written -- it was derived entirely
from check_sender_history's own docstring and type hint (sender_email: str),
exactly Module 1's real, unmodified function -- concrete proof that a
well-designed tool needs NO REWORK to become MCP-compatible.

Module 2 complete. Run Module 3 next.


### Module 3: Verifying the Server Actually Works — Calling the Real Tool Through MCP

Invoke the real tool through the server's own call_tool mechanism, confirming the wrapped version produces identical results to the direct call from Module 1.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Actually CALLING the tool through the real MCP server
# ------------------------------------------------------------------

async def call_tool_via_mcp(sender_email: str):
    result = await mcp_server.call_tool("check_sender_history", {"sender_email": sender_email})
    return result


print("=" * 70)
print("MODULE 3: CALLING THE REAL TOOL THROUGH THE MCP SERVER")
print("=" * 70)

mcp_result = asyncio.run(call_tool_via_mcp("shobha.chopra@email.com"))
print(f"\nResult via REAL MCP server.call_tool(): {mcp_result}")

# compare against the DIRECT function call from Module 1
direct_result = check_sender_history_impl("shobha.chopra@email.com")
print(f"\nResult via DIRECT function call (Module 1): {direct_result}")

# test a genuinely first-time sender through MCP too
mcp_result_new = asyncio.run(call_tool_via_mcp("first.time.sender@email.com"))
print(f"\nGenuinely first-time sender, via MCP: {mcp_result_new}")

print("\nCONFIRMED: this is a REAL, working MCP server -- built with the")
print("official Anthropic SDK, wrapping a real, previously-tested tool")
print("function with ZERO modifications, automatically deriving a correct")
print("schema, and correctly executing real tool calls through the actual")
print("protocol mechanics, not a simulation of them.")

print("\nModule 3 complete. All Chapter 11 Topic 4 modules done.")
print()
print("=" * 70)
print("CHAPTER 11 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  This used the REAL, official Anthropic MCP Python SDK (package: mcp,
  v1.28.1) -- not a simulation. @mcp.tool() genuinely works as described.

  The wrapped function (check_sender_history_impl) is EXACTLY Chapter 11
  Topic 2's real, tested implementation -- ZERO modifications needed,
  concretely proving the "well-designed tools are MCP-ready" claim.

  Schema derivation is REAL and AUTOMATIC -- derived entirely from the
  function's docstring and type hints, verified by actually inspecting
  the server's list_tools() output.

  This remains a LEARNING/READINESS exercise -- Topic 3's concrete
  adoption triggers (genuine second consumer, process isolation,
  cross-team reuse) still need to be met before this replaces this
  project's current, correct direct-integration production approach.
""")


MODULE 3: CALLING THE REAL TOOL THROUGH THE MCP SERVER

Result via REAL MCP server.call_tool(): [TextContent(type='text', text='{\n  "sender_email": "shobha.chopra@email.com",\n  "is_repeat_sender": true,\n  "first_seen_date": "2026-07-09",\n  "total_email_count": 1,\n  "last_topic": "premature withdrawal penalty",\n  "has_unresolved_issue": true,\n  "last_contact_date": "2026-07-09"\n}', annotations=None, meta=None)]

Result via DIRECT function call (Module 1): {'sender_email': 'shobha.chopra@email.com', 'is_repeat_sender': True, 'first_seen_date': '2026-07-09', 'total_email_count': 1, 'last_topic': 'premature withdrawal penalty', 'has_unresolved_issue': True, 'last_contact_date': '2026-07-09'}

Genuinely first-time sender, via MCP: [TextContent(type='text', text='{\n  "sender_email": "first.time.sender@email.com",\n  "is_repeat_sender": false\n}', annotations=None, meta=None)]

CONFIRMED: this is a REAL, working MCP server -- built with the
official Anthropic SDK, wrapping a real, pr